### 1. 年份/交互数

In [4]:
import pandas as pd

file_path = "./movie_book_cdsr_processed/full_reviews.json.gz"

# 只读取前 5 行
sample = pd.read_json(
    file_path,
    lines=True,
    compression='gzip',
    nrows=5
)

print(sample.to_string())
print("\n列名：", sample.columns.tolist())

          user_id  item_id  timestamp domain
0  A3R5OBKS7OM2IR   143502 2013-01-17  movie
1  A3R5OBKS7OM2IR   143529 2013-10-02  movie
2   AH3QC2PC1VTGP   143561 2008-07-17  movie
3  A3LKP6WPMP9UKX   143588 2009-03-13  movie
4   AVIY68KEPQ5ZD   143588 2009-01-18  movie

列名： ['user_id', 'item_id', 'timestamp', 'domain']


In [ ]:
import pandas as pd
import numpy as np

file_path = "./movie_book_cdsr_processed/full_reviews.json.gz"   # 你的保存路径
chunksize = 500000

year_counts = pd.Series(dtype=int)

for chunk in pd.read_json(file_path, lines=True, compression='gzip', chunksize=chunksize):
    # 将 unix 时间戳转为年份
    chunk['year'] = pd.to_datetime(chunk['timestamp'], unit='s').dt.year
    year_counts = year_counts.add(chunk['year'].value_counts(), fill_value=0)

year_counts = year_counts.sort_index().astype(int)

print("各年份交互数量：")
for yr, cnt in year_counts.items():
    print(f"  {int(yr)} : {cnt}")

# 计算 2012 及之后的占比
total = year_counts.sum()
after_2012 = year_counts[year_counts.index >= 2013].sum()
print(f"\n总交互数: {total}")
print(f"2012年及之后: {after_2012}  ({after_2012/total:.1%})")

各年份交互数量：
  1996 : 37
  1997 : 12175
  1998 : 58187
  1999 : 128108
  2000 : 402259
  2001 : 381172
  2002 : 383595
  2003 : 391491
  2004 : 476788
  2005 : 655497
  2006 : 725314
  2007 : 947284
  2008 : 1011805
  2009 : 1219674
  2010 : 1371842
  2011 : 1795859
  2012 : 3359876
  2013 : 8182205
  2014 : 5611034

总交互数: 27114202
2012年及之后: 17153115  (63.3%)


### 2. 下载目前图片

In [6]:
import os
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [7]:
# 配置
SAVE_DIR = "./movie_book_cdsr_processed"
META_CSV = f"{SAVE_DIR}/item_meta.csv"               # 你已有的元数据
IMAGE_DIR = f"{SAVE_DIR}/images"                     # 图片保存目录
MAPPING_FILE = f"{SAVE_DIR}/item_image_map.json"     # 映射结果
NUM_THREADS = 64                                     # 并发数
TIMEOUT = 5                                   # 超时秒数
os.makedirs(IMAGE_DIR, exist_ok=True)

In [8]:
# 加载元数据，得到 item_id 和 imUrl
meta_df = pd.read_csv(META_CSV)

urls = meta_df[['item_id', 'imUrl']].dropna(subset=['imUrl'])
urls = urls[urls['imUrl'].str.startswith('http')]    # 过滤无效链接

# 定义下载函数
def download_image(row):
    item_id, url = row['item_id'], row['imUrl']
    img_path = os.path.join(IMAGE_DIR, f"{item_id}.jpg")

    # 如果已有图片且大小 >0，跳过（断点续传）
    if os.path.exists(img_path) and os.path.getsize(img_path) > 0:
        return item_id, img_path, True

    try:
        resp = requests.get(url, timeout=TIMEOUT, stream=True)
        if resp.status_code == 200:
            with open(img_path, 'wb') as f:
                for chunk in resp.iter_content(1024):
                    f.write(chunk)
            # 检查真的写入了图片
            if os.path.getsize(img_path) > 0:
                return item_id, img_path, True
    except Exception:
        pass

    # 下载失败，删除可能残留的空文件
    if os.path.exists(img_path):
        os.remove(img_path)
    return item_id, None, False

# 并发下载，并记录结果
results = {}
failed_items = set()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {executor.submit(download_image, row): row for _, row in urls.iterrows()}
    for future in tqdm(as_completed(futures), total=len(futures), desc="下载图片"):
        item_id, path, ok = future.result()
        if ok:
            results[item_id] = path
        else:
            failed_items.add(item_id)

# 保存映射文件
with open(MAPPING_FILE, 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"下载成功: {len(results)} 张, 失败: {len(failed_items)} 张")

下载图片: 100%|██████████| 95543/95543 [14:18<00:00, 111.24it/s]

下载成功: 95279 张, 失败: 264 张
